In [14]:
#uncomment to install
!pip -q install biopython

import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

!nvidia-smi

TensorFlow: 2.20.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Sun Jun 14 18:49:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             31W /   70W |     169MiB /  15360MiB |      0%      Default |
|                                     

In [15]:
import os
import math
import random
import numpy as np
import tensorflow as tf

from itertools import product
from Bio import SeqIO

from tensorflow.keras.models import Sequential, model_from_json
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Activation,
    Flatten,
    LSTM,
    Conv1D,
    MaxPooling1D,
)
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import EarlyStopping


In [16]:
MAX_LEN = 64

In [17]:

def read_fasta(data_path):
    records = list(SeqIO.parse(data_path, "fasta"))
    text = ""

    for record in records:
        seq = str(record.seq).upper()

        # Evita errors si hi ha N o altres caràcters no ACGT
        seq = "".join(c for c in seq if c in chars)

        text += seq

    if len(text) <= MAX_LEN:
        raise ValueError(
            f"La seqüència de {data_path} és massa curta: {len(text)} bases."
        )

    return text


def read_data(data_path):
    text = read_fasta(data_path)

    for i in range(0, len(text) - MAX_LEN, step):
        sentence = text[i : i + MAX_LEN]
        next_char = text[i + MAX_LEN]
        yield sentence, next_char


def vectorization(sentences, next_chars):
    current_batch_size = len(sentences)

    x = np.zeros((current_batch_size, maxlen, len(chars)), dtype=bool)
    y = np.zeros((current_batch_size, len(chars)), dtype=bool)

    for i, sentence in enumerate(sentences):
        for t, char in enumerate(sentence):
            x[i, t, char_indices[char]] = 1

        y[i, char_indices[next_chars[i]]] = 1

    return x, y


def get_batch(stream):
    sentences = []
    next_chars = []

    for sentence, next_char in stream:
        sentences.append(sentence)
        next_chars.append(next_char)

        if len(sentences) == batch_size:
            yield vectorization(sentences, next_chars)
            sentences = []
            next_chars = []

    # També aprofitem l’últim batch encara que sigui més petit
    if sentences:
        yield vectorization(sentences, next_chars)



In [18]:
def my_kernel_initializer(shape, dtype=None):
    x = np.zeros(shape, dtype=bool)

    for i, c in enumerate(product("ACGT", repeat=5)):
        kmer = c * 3
        for t, char in enumerate(kmer):
            x[t, char_indices[char], i] = 1

    return x

In [19]:


def model_CNN_LSTM():
    print("Build model...")

    model = Sequential([
        Input(shape=(MAX_LEN, input_dim)),

        Conv1D(
            filters=1024,
            kernel_size=24,
            trainable=True,
            padding="valid",
            activation="relu",
            strides=1,
        ),

        MaxPooling1D(pool_size=3),
        Dropout(0.1),

        LSTM(256, return_sequences=True),
        Dropout(0.2),

        Flatten(),

        Dense(1024),
        Activation("relu"),

        Dense(input_dim),
        Activation("softmax"),
    ])

    optimizer = RMSprop(learning_rate=0.001)
    model.compile(loss="categorical_crossentropy", optimizer=optimizer)

    return model


In [20]:

def save_model(epoch, model):
    os.makedirs("./model", exist_ok=True)

    # Arquitectura JSON
    with open("./model/model.json", "w") as json_file:
        json_file.write(model.to_json())

    # Pesos per epoch
    weights_path = f"./model/model_{epoch}.weights.h5"
    model.save_weights(weights_path)

    # Últim checkpoint
    model.save_weights("./model/model_latest.weights.h5")

    # Model complet, més còmode per recarregar
    model.save(f"./model/model_{epoch}.keras")
    model.save("./model/model_latest.keras")

    print(f"Saved model to disk: {weights_path}")


def load_model_from_json(weights_path="./model/model_latest.weights.h5"):
    with open("./model/model.json", "r") as json_file:
        loaded_model_json = json_file.read()

    model = model_from_json(loaded_model_json)
    model.load_weights(weights_path)

    optimizer = RMSprop(learning_rate=0.001)
    model.compile(loss="categorical_crossentropy", optimizer=optimizer)

    print("Loaded model from disk")

    return model



In [21]:

def evaluate_entropy(model, data_path):
    print("----- Testing entropy")

    losses = []

    for batch in get_batch(read_data(data_path)):
        x_batch, y_batch = batch
        loss = model.test_on_batch(x_batch, y_batch)
        losses.append(float(loss))

    if not losses:
        raise ValueError(f"No hi ha batches per avaluar a {data_path}")

    # Converteix de nats a bits
    return float(np.mean(losses)) * math.log(math.e, 2)



In [22]:
def print_gpu_memory(tag=""):
    try:
        info = tf.config.experimental.get_memory_info("GPU:0")
        current_gb = info["current"] / 1024**3
        peak_gb = info["peak"] / 1024**3
        print(f"[GPU MEM] {tag} current={current_gb:.2f} GB | peak={peak_gb:.2f} GB")
    except Exception as e:
        print(f"[GPU MEM] No s'ha pogut llegir la memòria GPU: {e}")

In [23]:
BASE_DIR = "/content/drive/MyDrive/TFG/train_data/splitted"

# Reproducibilitat
np.random.seed(1337)
tf.random.set_seed(1337)
random.seed(1337)


# Paths
train_path = os.path.join(BASE_DIR, "train_1.fasta")
valid_path = os.path.join(BASE_DIR, "valid_1.fasta")
test_path  = os.path.join(BASE_DIR, "test_1.fasta")


for path in [train_path, valid_path, test_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"No trobo aquest fitxer: {path}")


chars = "ACGT"
print("total chars:", len(chars))
print("chars:", chars)

char_indices = {c: i for i, c in enumerate(chars)}
indices_char = {i: c for i, c in enumerate(chars)}

maxlen = 64
step = 1
batch_size = 64
epochs = 10
input_dim = len(chars)

early_stopping = EarlyStopping(monitor="val_loss", patience=2)

total chars: 4
chars: ACGT


In [29]:
model = model_CNN_LSTM()
model.summary()


Build model...


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)               │ (None, 41, 1024)       │        99,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 13, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 13, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 13, 256)        │     1,311,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 13, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 3328)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 1024)           │     3,408,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_12 (Activation)      │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 4)              │         4,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_13 (Activation)      │ (None, 4)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,824,068 (18.40 MB)

 Trainable params: 4,824,068 (18.40 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Tria model
model = model_CNN_LSTM()
model.summary()

print_gpu_memory("després de crear el model")

entropy = []

for epoch in range(epochs):
    print("this is epoch:", epoch)

    # Opcional: reinicia el comptador de pic al principi de cada epoch
    try:
        tf.config.experimental.reset_memory_stats("GPU:0")
    except Exception:
        pass

    for i, batch in enumerate(get_batch(read_data(train_path))):
        x_batch, y_batch = batch

        loss = model.train_on_batch(x_batch, y_batch)

        if i % 200 == 0:
            print(epoch, "\tbatch:", i, "\tloss bits:", float(loss) * math.log(math.e, 2))
            print_gpu_memory(f"epoch {epoch}, batch {i}")

    print_gpu_memory(f"final epoch {epoch}")

    save_model(epoch, model)

    valid_entropy = evaluate_entropy(model, valid_path)
    print("valid entropy:", valid_entropy)

    print_gpu_memory(f"després validació epoch {epoch}")

    entropy.append(valid_entropy)

Build model...


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_1 (Conv1D)               │ (None, 41, 1024)       │        99,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 13, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 13, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 13, 256)        │     1,311,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 13, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3328)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1024)           │     3,408,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │         4,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 4)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,824,068 (18.40 MB)

 Trainable params: 4,824,068 (18.40 MB)

 Non-trainable params: 0 (0.00 B)

[GPU MEM] després de crear el model current=0.04 GB | peak=0.07 GB
this is epoch: 0
0 	batch: 0 	loss bits: 2.0038690991921313
[GPU MEM] epoch 0, batch 0 current=0.06 GB | peak=2.22 GB
0 	batch: 200 	loss bits: 1.9864761497450736
[GPU MEM] epoch 0, batch 200 current=0.06 GB | peak=2.22 GB
0 	batch: 400 	loss bits: 1.9594153675624142
[GPU MEM] epoch 0, batch 400 current=0.06 GB | peak=2.22 GB
0 	batch: 600 	loss bits: 1.947815997674972
[GPU MEM] epoch 0, batch 600 current=0.06 GB | peak=2.22 GB
0 	batch: 800 	loss bits: 1.9419911172728996
[GPU MEM] epoch 0, batch 800 current=0.06 GB | peak=2.22 GB
0 	batch: 1000 	loss bits: 1.9375758066772988
[GPU MEM] epoch 0, batch 1000 current=0.06 GB | peak=2.22 GB
0 	batch: 1200 	loss bits: 1.9344899219728673
[GPU MEM] epoch 0, batch 1200 current=0.06 GB | peak=2.22 GB
0 	batch: 1400 	loss bits: 1.9281850379922367
[GPU MEM] epoch 0, batch 1400 current=0.06 GB | peak=2.22 GB
0 	batch: 1600 	loss bits: 1.9202200054828276
[GPU MEM] epoch 0, batch 1600

KeyboardInterrupt: 

In [ ]:
# Tria model
model = model_CNN_LSTM()
model.summary()


entropy = []

for epoch in range(epochs):
    print("this is epoch:", epoch)

    for i, batch in enumerate(get_batch(read_data(train_path))):
        x_batch, y_batch = batch

        loss = model.train_on_batch(x_batch, y_batch)

        if i % 200 == 0:
            print(epoch, "\tbatch:", i, "\tloss bits:", float(loss) * math.log(math.e, 2))

    save_model(epoch, model)

    valid_entropy = evaluate_entropy(model, valid_path)
    print("valid entropy:", valid_entropy)

    entropy.append(valid_entropy)


print("Validation entropy per epoch:")
print(entropy)


test_entropy = evaluate_entropy(model, test_path)
print("Test entropy:", test_entropy)

FileNotFoundError: No trobo aquest fitxer: /content/fasta_split/train_1.fasta